In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "excel_features_clean.csv"

data = pd.read_csv(DATA_PATH)

print(data.shape)
data.head()

(89, 34)


,العمر,النوع_أنثي,النوع_ذكر,الصف الدراسي_الأول إعدادى,الصف الدراسي_الثالث إعدادى,الصف الدراسي_الرابع الإبتدائي,الصف الدراسي_السادس الإبتدائي,التسلسل_تسلسل ثابت,التسلسل_تسلسل غير منظم,التسلسل_تسلسل مرتبك,...,gestalt_تجزئة الجشطالت,gestalt_صعوبة رسم الأشكال المتداخلة,note_الرسم من أسفل لأعلى,note_استخدام أكثر من اتجاه,note_المداومة,note_إعادة رسم الأجزاء (خطوط ثقيلة),note_اندفاع وسرعة في الرسم,note_رسم فواصل بين الأشكال,note_رسم كاريكاتيري,target
0,9,0,1,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,سوي
1,9,0,1,0,0,1,0,0,0,0,...,0,1,0,0,0,1,0,0,0,سوي
2,9,0,1,0,0,1,0,0,1,0,...,0,1,0,0,1,0,1,0,0,سوي
3,10,0,1,0,0,1,0,0,1,0,...,0,1,0,0,1,0,0,0,0,سوي
4,11,0,1,0,0,1,0,0,1,0,...,0,1,0,0,0,0,0,0,0,سوي


In [6]:
X = data.drop(columns=["target"])
y = data["target"]

print("X shape:", X.shape)
print("Target:")
print(y.value_counts())

X shape: (89, 33)
Target:
target
سوي              84
العصاب            3
التخلف العقلي     2
Name: count, dtype: int64


In [7]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)
print("Encoded target:", np.unique(y_encoded))

Classes: ['التخلف العقلي' 'العصاب' 'سوي']
Encoded target: [0 1 2]


### Stratified K-Fold for split dataset

In [8]:
## what is stratifed? -> the problem is that if we have imbalanced classes, 
# we want to make sure that each fold has a similar distribution of classes. 
# StratifiedKFold ensures that the proportion of classes in each fold is similar to the overall distribution in the dataset.
# This is important for training and evaluating models, especially when dealing with imbalanced datasets,
# as it helps to prevent bias in the model's performance evaluation
# like if we have a fold that have only one class this is a disaster bc our model will be trained on olny a one class.

## in the Kfold, it will not perserve the distribution of classes in each fold like in our main dataset.

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            class_weight="balanced",
            random_state=42
        ))
    ])
}

In [10]:
results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y_encoded,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        return_train_score=False
    )
    
    results.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std()
    })

results_df = pd.DataFrame(results)
results_df

d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std
0,Logistic Regression,0.943791,0.035161,0.554897,0.229786
1,Random Forest,0.943791,0.001307,0.485541,0.000346
2,SVM,0.954902,0.022585,0.588398,0.205801


In [11]:
best_model = models["SVM"]

y_pred = cross_val_predict(
    best_model,
    X,
    y_encoded,
    cv=cv
)

print(classification_report(
    y_encoded,
    y_pred,
    target_names=label_encoder.classes_
))

print(confusion_matrix(y_encoded, y_pred))

               precision    recall  f1-score   support

التخلف العقلي       0.00      0.00      0.00         2
       العصاب       1.00      0.33      0.50         3
          سوي       0.95      1.00      0.98        84

     accuracy                           0.96        89
    macro avg       0.65      0.44      0.49        89
 weighted avg       0.93      0.96      0.94        89

[[ 0  0  2]
 [ 0  1  2]
 [ 0  0 84]]


d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\FCAI\Year 3\2nd semster\Cognitive\b

In [12]:
cm = confusion_matrix(y_encoded, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=label_encoder.classes_,
    columns=label_encoder.classes_
)

cm_df

,التخلف العقلي,العصاب,سوي
التخلف العقلي,0,0,2
العصاب,0,1,2
سوي,0,0,84


In [14]:
results_path = PROJECT_ROOT / "data" / "processed" / "excel_ml_results.csv"

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")

print("Saved:", results_path)
results_df

Saved: d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\data\processed\excel_ml_results.csv


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std
0,Logistic Regression,0.943791,0.035161,0.554897,0.229786
1,Random Forest,0.943791,0.001307,0.485541,0.000346
2,SVM,0.954902,0.022585,0.588398,0.205801
